# 💳 CodeAlpha — Credit Scoring Model

**Task 1:** Predict creditworthiness as a binary classification problem.

We will learn while building: data inspection → EDA → preprocessing → classification → evaluation → model selection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'data' / 'german_credit.csv'

In [ ]:
URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'
columns = ['checking_status','duration_months','credit_history','purpose','credit_amount','savings_status','employment','installment_rate','personal_status_sex','other_debtors','residence_since','property','age','other_installment_plans','housing','existing_credits','job','dependents','telephone','foreign_worker','target']
df = pd.read_csv(URL, sep=r'\s+', header=None, names=columns)
df['target'] = df['target'].map({1:1,2:0})
df.to_csv(DATA_PATH, index=False)
df.head()

## 1. Understand the data

- **Features (X):** customer credit-history attributes.
- **Target (y):** 1 = good credit, 0 = bad credit.
- This is **classification**, not regression.

In [ ]:
print('Shape:', df.shape)
display(df.head())
df.info()
print('\nMissing values:\n', df.isna().sum().sort_values(ascending=False).head())

In [ ]:
print(df['target'].value_counts())
sns.countplot(data=df, x='target')
plt.title('Creditworthiness Class Distribution')
plt.show()

## 2. Train/test split and preprocessing

Categorical columns need **One-Hot Encoding**. Numerical columns are standardized for Logistic Regression. A pipeline keeps preprocessing and the model together and helps prevent data leakage.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

X = df.drop(columns='target')
y = df['target']
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
preprocessor = ColumnTransformer([('num', StandardScaler(), num_cols), ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

models = {'Logistic Regression': LogisticRegression(max_iter=2000), 'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42), 'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42)}
results=[]
for name, model in models.items():
    pipe=Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test); prob=pipe.predict_proba(X_test)[:,1]
    results.append([name, accuracy_score(y_test,pred), precision_score(y_test,pred,zero_division=0), recall_score(y_test,pred,zero_division=0), f1_score(y_test,pred,zero_division=0), roc_auc_score(y_test,prob)])
results_df=pd.DataFrame(results,columns=['Model','Accuracy','Precision','Recall','F1','ROC_AUC']).sort_values('ROC_AUC',ascending=False)
results_df

## 3. Select and save the best model

For this project we compare the models using ROC-AUC and keep the highest-scoring pipeline.

In [ ]:
import joblib
best_name = results_df.iloc[0]['Model']
best_model = models[best_name]
best_pipeline = Pipeline([('preprocessor', preprocessor), ('model', best_model)])
best_pipeline.fit(X_train,y_train)
joblib.dump(best_pipeline, ROOT/'models'/'credit_scoring_model.joblib')
print('Best model:', best_name)
print('Saved model:', ROOT/'models'/'credit_scoring_model.joblib')

## 4. Conclusion

We built a complete classification workflow: inspection, EDA, encoding, scaling, train/test split, three classifiers, and evaluation using Accuracy, Precision, Recall, F1-score and ROC-AUC. The saved pipeline can be used by the Streamlit app.